# Topic 1 - examples

Exploring example-sentence sources at the granularity each one actually
supports. See [`05.54_data_enrich.md`](../../scratch_space/09_concept_model/05.54_data_enrich/05.54_data_enrich.md) Topic 1.

Open questions: OMW example coverage per language; whether non-en lexicons
carry their own examples; how much Tatoeba would add and at what
sense-blindness cost.

## Setup

Thin caller over the staged cache and the OMW wordnets.

In [ ]:
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import wn
from loguru import logger as lg

from lang_tools.lexicon.ingestion.sources.omw import OMW_LEXICONS, OMW_VERSION
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = ["en", "pt", "es", "fr", "it"]
paths = get_lang_tools_params().paths
data_fol = paths.data_fol
staging = data_fol / "_raw/lexicon/staging"
wn.config.data_directory = str(data_fol / "_raw/lexicon/wn_data")


def staged(dataset: str, lang: str) -> pd.DataFrame:
    """Read a staged parquet (``<staging>/<dataset>/<lang>.parquet``)."""
    return pq.read_table(staging / dataset / f"{lang}.parquet").to_pandas()


def wordnet(lang: str) -> wn.Wordnet:
    """Open the pinned OMW lexicon for a language (one wordnet, no merging)."""
    return wn.Wordnet(lexicon=f"{OMW_LEXICONS[lang]}:{OMW_VERSION}")


def ili_of(synset: wn.Synset) -> str | None:
    """Return the synset's ILI id as a plain string, or ``None``."""
    il = synset.ili
    return getattr(il, "id", il) or None


lg.info("staging at {}", staging)

## OMW example + definition coverage

In [ ]:
# OMW example + definition coverage per language (one pass per wordnet).
rows = []
for lang in LANGS:
    n = n_ex = n_def = 0
    per_synset = []
    for s in wordnet(lang).synsets():
        n += 1
        k = len(s.examples())
        per_synset.append(k)
        if k:
            n_ex += 1
        if s.definition():
            n_def += 1
    arr = np.array(per_synset)
    rows.append(
        {
            "lang": lang,
            "synsets": n,
            "with_example": n_ex,
            "example_pct": round(100 * n_ex / n, 1),
            "with_definition": n_def,
            "max_ex_per_synset": int(arr.max()),
        }
    )
omw_examples = pd.DataFrame(rows).set_index("lang")
omw_examples

## Tatoeba supply (sense-blind lemma join)

In [ ]:
# Tatoeba: raw sentence supply per language (the lemma join is sense-blind).
tat = pd.DataFrame(
    {"lang": LANGS, "sentences": [len(staged("tatoeba", lang)) for lang in LANGS]}
).set_index("lang")
# How many distinct en OMW lemma forms exist (the join target for examples)?
en_forms = {w.lemma().lower() for w in wordnet("en").words()}
print("distinct en OMW lemma forms:", len(en_forms))
tat

## Findings (measured 2026-06-21)

- **OMW examples are English-only and partial.** en 27% of synsets carry an
  example (32,917 / 117,659); it 4%; pt / es / fr 0%. Definitions follow the
  same shape: en 100%, it 6%, pt / es / fr 0%.
- Because every synset is ILI-linked (100% in all five lexicons, see Topic 2),
  an English example attaches to a **concept** and can fan out to that
  concept's lemmas in every language. Examples are concept-level, sourced from
  English, not per-language.
- **Tatoeba supplies far more raw text** (en 2.03M, it 0.97M, fr 0.72M sentences)
  but the lemma join is sense-blind, and it is CC-BY (attribution cost).

**Decision for Step 4:** ship examples from OMW only this phase, as a
concept-level field keyed on ILI, fanned out to senses. Defer Tatoeba; if wired
later it must carry a sense-blind flag and the CC-BY attribution. Non-en
example coverage from OMW is effectively zero, so do not expect per-language
examples.